# Modelos lineales con Bambi

Vamos a escribir las cuatro estructuras de la notebook anterior con Bambi, obtener predicciones y comparar los modelos. Usamos los mismos datos y los _priors_ automáticos de Bambi, que se muestran al crear cada modelo.

In [ ]:
# Instalación opcional en Google Colab
import sys

if "google.colab" in sys.modules:
    %pip install -q "pymc==6.4.0" "preliz==0.28.0" "arviz-base==1.3.1" "matplotlib==3.11.2" "nutpie==0.16.11"
    %pip install -q "git+https://github.com/bambinos/bambi.git@dev"

In [ ]:
import arviz as az
import bambi as bmb
from bambi import interpret
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 2026
COLORES = ["#0072B2", "#E69F00", "#009E73", "#CC79A7"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=COLORES)

## Datos y fórmulas

Usamos `kid_score` para la puntuación del niño, `mom_iq` para el IQ materno y `mom_hs` para indicar si la madre terminó la secundaria. Bambi recibe los datos en un `DataFrame` de pandas. Declaramos `mom_hs` como categórica para que genere los coeficientes correspondientes a los grupos.

In [ ]:
url_datos = "https://raw.githubusercontent.com/avehtari/ROS-Examples/master/KidIQ/data/kidiq.csv"
datos = pd.read_csv(url_datos)
grupos = ["No terminó", "Terminó"]
datos["mom_hs"] = pd.Categorical(datos["mom_hs"].map({0: grupos[0], 1: grupos[1]}), categories=grupos)
datos = datos[["kid_score", "mom_iq", "mom_hs"]]
datos.head()

Las fórmulas son muy similares a las que se usan en R. `1` incluye un intercepto, `0` lo retira y `:` agrega una interacción.

| Modelo | Fórmula |
| --- | --- |
| Sin covariables | `kid_score ~ 1` |
| Intercepto y pendiente comunes | `kid_score ~ 1 + mom_iq` |
| Intercepto variable por grupo | `kid_score ~ 0 + mom_hs + mom_iq` |
| Intercepto y pendiente variables por grupo | `kid_score ~ 0 + mom_hs + mom_iq + mom_hs:mom_iq` |

En todos los casos usamos la familia `gaussian`. Vamos directamente a ajustar y graficar.

## Sin covariables

Creamos el modelo y miramos los _priors_ que eligió Bambi.

In [ ]:
modelo_0 = bmb.Model("kid_score ~ 1", datos, family="gaussian")
modelo_0

In [ ]:
idata_0 = modelo_0.fit(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_0, kind="all", ci_kind="eti", ci_prob=0.95, round_to=2)

Con `predict(kind="response")` generamos puntuaciones predictivas. Las guarda en `posterior_predictive`, que podemos usar directamente con ArviZ.

In [ ]:
modelo_0.predict(idata_0, kind="response", random_seed=SEMILLA)
az.plot_ppc_dist(idata_0, var_names="kid_score", num_samples=50)
ax = plt.gcf().axes[0]
ax.set(xlabel="Puntuación del niño", ylabel="Densidad");

## Intercepto y pendiente comunes

Agregamos `mom_iq` a la fórmula.

In [ ]:
modelo_1 = bmb.Model("kid_score ~ 1 + mom_iq", datos, family="gaussian")
modelo_1

In [ ]:
idata_1 = modelo_1.fit(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(
    idata_1,
    var_names=["Intercept", "mom_iq", "sigma"],
    kind="all",
    ci_kind="eti",
    ci_prob=0.95,
    round_to=2
)

`interpret.plot_predictions()` calcula y dibuja las predicciones sin armar las bandas a mano. Con `conditional="mom_iq"` evalúa una grilla de IQ. Por defecto muestra la media condicional. Pedimos intervalos del 50% y del 95% con `prob=[0.5, 0.95]`.

In [ ]:
figura_1 = interpret.plot_predictions(
    modelo_1,
    idata_1,
    conditional="mom_iq",
    prob=[0.5, 0.95],
    fig_kwargs={"xlabel": "IQ de la madre", "ylabel": "Puntuación media del niño"},
)
figura_1

La función devuelve una figura de Matplotlib. Podemos reutilizarla para agregar los datos, cambiar el título y ajustar los límites.

In [ ]:
ax = figura_1.axes[0]
ax.scatter(datos["mom_iq"], datos["kid_score"], color="0.4", s=15, alpha=0.35, zorder=0)
ax.set(title="Media condicional e IC del 50% y 95%", xlim=(65, 140))
figura_1

Para obtener las bandas de un niño nuevo usamos `target="kid_score"`. Con `on=` podemos elegir dónde dibujar: acá reutilizamos los ejes de una figura con dos paneles para comparar la incertidumbre sobre la media y sobre una nueva puntuación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True, layout="constrained")
targets = ["mean", "kid_score"]
titulos = ["Media condicional", "Puntuación de un niño nuevo"]
for ax, target, titulo in zip(axes, targets, titulos):
    interpret.plot_predictions(
        modelo_1,
        idata_1,
        conditional="mom_iq",
        target=target,
        prob=[0.5, 0.95],
        on=ax,
        fig_kwargs={"xlabel": "IQ de la madre", "ylabel": "Puntuación del niño"},
    )
    ax.scatter(datos["mom_iq"], datos["kid_score"], color="0.4", s=12, alpha=0.25, zorder=0)
    ax.set(title=titulo, xlabel="IQ de la madre", ylabel="Puntuación del niño")

Si queremos una tabla en lugar de un gráfico, usamos `interpret.predictions()`. Su atributo `.summary` ya trae las medias y los límites de los intervalos. Podemos cambiar la lista de IQ sin volver a ajustar.

In [ ]:
iq_nuevos = [80.0, 100.0, 120.0]
preds_1 = interpret.predictions(
    modelo_1, idata_1, conditional={"mom_iq": iq_nuevos}, prob=[0.5, 0.95]
)
preds_1.summary

## Intercepto variable

`0 + mom_hs` crea un intercepto para cada grupo y `mom_iq` mantiene una pendiente común.

In [ ]:
modelo_2 = bmb.Model("kid_score ~ 0 + mom_hs + mom_iq", datos, family="gaussian")
modelo_2

In [ ]:
idata_2 = modelo_2.fit(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_2, kind="all", ci_kind="eti", ci_prob=0.95, round_to=3)

Para ver cada grupo por separado, indicamos `panel="mom_hs"` en `subplot_kwargs`. Bambi calcula las bandas y arma los paneles.

In [ ]:
figura_2 = interpret.plot_predictions(
    modelo_2,
    idata_2,
    conditional=["mom_iq", "mom_hs"],
    prob=[0.5, 0.95],
    subplot_kwargs={"main": "mom_iq", "panel": "mom_hs"},
    fig_kwargs={
        "xlabel": "IQ de la madre",
        "ylabel": "Puntuación media del niño",
        "theme": {"figure.figsize": (10, 4)}
    },
)

for ax, grupo in zip(figura_2.axes, grupos):
    seleccion = datos["mom_hs"] == grupo
    ax.scatter(
        datos.loc[seleccion, "mom_iq"],
        datos.loc[seleccion, "kid_score"],
        color="0.5",
        s=14,
        alpha=0.35,
        zorder=0,
    )
    ax.set(title=grupo, xlabel="IQ de la madre", ylabel="Puntuación media del niño")

figura_2

## Intercepto y pendiente variables

Agregamos `mom_hs:mom_iq` para que las pendientes también puedan cambiar por grupo. Con esta fórmula, `mom_iq` es la pendiente del grupo de referencia y `mom_hs:mom_iq` representa el cambio de pendiente para el otro grupo.

In [ ]:
modelo_3 = bmb.Model("kid_score ~ 0 + mom_hs + mom_iq + mom_hs:mom_iq", datos, family="gaussian")
modelo_3

In [ ]:
idata_3 = modelo_3.fit(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_3, kind="all", ci_kind="eti", ci_prob=0.95, round_to=2)

También se puede escribir `kid_score ~ 1 + mom_hs + mom_iq + mom_hs:mom_iq`, o su abreviación `kid_score ~ mom_hs * mom_iq`. Con esas variantes, el intercepto corresponde al grupo de referencia y el coeficiente de `mom_hs` es una diferencia de interceptos. Las fórmulas representan las mismas rectas posibles, pero cambian la interpretación de los coeficientes. Al cambiar la parametrización también conviene revisar los _priors_.

In [ ]:
figura_3 = interpret.plot_predictions(
    modelo_3,
    idata_3,
    conditional=["mom_iq", "mom_hs"],
    prob=[0.5, 0.95],
    subplot_kwargs={"main": "mom_iq", "panel": "mom_hs"},
    fig_kwargs={
        "xlabel": "IQ de la madre",
        "ylabel": "Puntuación media del niño",
        "theme": {"figure.figsize": (10, 4)}
    },
)

for ax, grupo in zip(figura_3.axes, grupos):
    seleccion = datos["mom_hs"] == grupo
    ax.scatter(
        datos.loc[seleccion, "mom_iq"],
        datos.loc[seleccion, "kid_score"],
        color="0.5",
        s=14,
        alpha=0.35,
        zorder=0,
    )
    ax.set(title=grupo, xlabel="IQ de la madre", ylabel="Puntuación media del niño")

figura_3

Podemos obtener las predicciones para ambos grupos y varios IQ en una sola llamada.

In [ ]:
preds_3 = interpret.predictions(
    modelo_3,
    idata_3,
    conditional={"mom_iq": iq_nuevos, "mom_hs": grupos},
    prob=[0.5, 0.95],
)
preds_3.summary

Para comparar los grupos a igual IQ, `interpret.comparisons()` calcula directamente la diferencia entre sus medias. El orden del contraste indica que restamos «No terminó» a «Terminó».

In [ ]:
diferencias_3 = interpret.comparisons(
    modelo_3,
    idata_3,
    contrast={"mom_hs": grupos},
    conditional={"mom_iq": iq_nuevos},
    prob=[0.5, 0.95],
)
diferencias_3.summary

In [ ]:
figura_diferencias = interpret.plot_comparisons(
    modelo_3,
    idata_3,
    contrast={"mom_hs": grupos},
    conditional="mom_iq",
    prob=[0.5, 0.95],
    fig_kwargs={
        "xlabel": "IQ de la madre",
        "ylabel": "Diferencia de medias (Terminó − No terminó)",
    },
)
figura_diferencias.axes[0].axhline(0, color="0.4", ls="--")
figura_diferencias

## Comparación de modelos

Primero generamos réplicas con las covariables observadas y comparamos sus distribuciones con las puntuaciones de los niños.

In [ ]:
modelos = {
    "Sin covariables": modelo_0,
    "Intercepto y pendiente comunes": modelo_1,
    "Intercepto variable": modelo_2,
    "Intercepto y pendiente variables": modelo_3,
}
resultados = {
    "Sin covariables": idata_0,
    "Intercepto y pendiente comunes": idata_1,
    "Intercepto variable": idata_2,
    "Intercepto y pendiente variables": idata_3,
}

for nombre, modelo in modelos.items():
    modelo.predict(resultados[nombre], kind="response", random_seed=SEMILLA)

In [ ]:
for nombre, idata in resultados.items():
    az.plot_ppc_dist(idata, var_names="kid_score", num_samples=50)
    ax = plt.gcf().axes[0]
    ax.set(title=nombre, xlabel="Puntuación del niño", ylabel="Densidad")

Para comparar la capacidad predictiva con LOO, agregamos las logverosimilitudes usando el método de Bambi y pasamos los resultados a ArviZ.

In [ ]:
for nombre, modelo in modelos.items():
    modelo.compute_log_likelihood(resultados[nombre])

In [ ]:
resultados_loo = {nombre: az.loo(idata, pointwise=True) for nombre, idata in resultados.items()}
comparacion = az.compare(resultados_loo, round_to=2)
comparacion

In [ ]:
az.plot_compare(comparacion);

La tabla ordena los modelos por su capacidad predictiva estimada. Miramos `elpd_diff` junto con `dse` para evaluar el tamaño y la incertidumbre de las diferencias. Si aparecieran advertencias de Pareto $k$, revisamos esos casos antes de interpretar la comparación.